In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.naive_bayes import GaussianNB
from sklearn.tree import ExtraTreeClassifier
from sklearn.multiclass import OneVsRestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score
)

import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "district_targeted.csv"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "outputs"
)

FIGURES_DIR = OUTPUT_DIR / "figures"
TABLES_DIR = OUTPUT_DIR / "tables"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Dataset:", DATA_PATH)

Project root: d:\Major_Project\Crime_Analysis
Dataset: d:\Major_Project\Crime_Analysis\data\processed\district_targeted.csv


In [2]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
print("\nColumns:", len(df.columns))

df.head()

Dataset shape: (9856, 333)

Columns: 333


,STATE,UNIT_NAME,YEAR,IPC_MURDER,IPC_ATTEMPT_TO_MURDER,IPC_RAPE,IPC_OTHER_RAPE,IPC_KIDNAPPING_AND_ABDUCTION_OF_WOMEN_AND_GIRLS,IPC_KIDNAPPING_AND_ABDUCTION_OF_OTHERS,IPC_DACOITY,...,IPC_CAUSING_DEATH_BY_NEGLIGENCE,IPC_CRIMINAL_BREACH_OF_TRUST,IPC_CRUELTY_BY_HUSBAND_OR_HIS_RELATIVES,IPC_CULPABLE_HOMICIDE_NOT_AMOUNTING_TO_MURDER,IPC_CUSTODIAL_RAPE,IPC_DOWRY_DEATHS,IPC_KIDNAPPING_ABDUCTION,IPC_OTHER_IPC_CRIMES,VIOLENT_CRIME_BURDEN,TARGET_VIOLENT_LEVEL
0,ANDHRA PRADESH,ADILABAD,2001,101.0,60.0,50.0,50.0,30.0,16.0,9.0,...,181.0,16.0,175.0,17.0,0.0,16.0,46.0,1518.0,544.0,4.0
1,ANDHRA PRADESH,ANANTAPUR,2001,151.0,125.0,23.0,23.0,30.0,23.0,8.0,...,270.0,11.0,154.0,1.0,0.0,7.0,53.0,754.0,697.0,5.0
2,ANDHRA PRADESH,CHITTOOR,2001,101.0,57.0,27.0,27.0,34.0,25.0,4.0,...,404.0,33.0,186.0,2.0,0.0,14.0,59.0,1262.0,558.0,4.0
3,ANDHRA PRADESH,CUDDAPAH,2001,80.0,53.0,20.0,20.0,20.0,5.0,1.0,...,233.0,12.0,57.0,1.0,0.0,17.0,25.0,1181.0,433.0,4.0
4,ANDHRA PRADESH,EAST GODAVARI,2001,82.0,67.0,23.0,23.0,26.0,23.0,4.0,...,431.0,50.0,247.0,1.0,0.0,12.0,49.0,2313.0,446.0,4.0


In [3]:
TARGET = "TARGET_VIOLENT_LEVEL"

before = len(df)

df = df.dropna(
    subset=[TARGET]
).copy()

df[TARGET] = (
    df[TARGET]
    .astype(int)
)

print("Rows before removing missing target:", before)
print("Rows after:", len(df))
print("Removed:", before - len(df))

print("\nTarget distribution:")
print(
    df[TARGET]
    .value_counts()
    .sort_index()
)

Rows before removing missing target: 9856
Rows after: 9631
Removed: 225

Target distribution:
TARGET_VIOLENT_LEVEL
1    1927
2    1926
3    1926
4    1926
5    1926
Name: count, dtype: int64


In [4]:
TARGET_SOURCE_COLUMNS = [
    "IPC_CULPABLE_HOMICIDE_NOT_AMOUNTING_TO_MURDER",

    "IPC_RIOTS",
    "IPC_RIOTS_COMMUNAL",
    "IPC_RIOTS_INDUSTRIAL",
    "IPC_RIOTS_POLITICAL",
    "IPC_RIOTS_CASTE_CONFLICT",
    "IPC_RIOTS_AGRARIAN",
    "IPC_RIOTS_STUDENTS",
    "IPC_RIOTS_SECTARIAN",

    "IPC_RAPE",
    "IPC_OTHER_RAPE",
    "IPC_CUSTODIAL_RAPE",
    "IPC_CUSTODIAL_GANG_RAPE",
    "IPC_RAPE_GANG_RAPE",
    "IPC_RAPE_OTHERS",

    "IPC_MURDER",

    "IPC_PREPARATION_AND_ASSEMBLY_FOR_DACOITY",

    "IPC_DOWRY_DEATHS",

    "IPC_ROBBERY",

    "IPC_DACOITY",
    "IPC_OTHER_DACOITY",
    "IPC_DACOITY_WITH_MURDER",

    "IPC_KIDNAPPING_AND_ABDUCTION_OF_WOMEN_AND_GIRLS",
    "IPC_KIDNAPPING_AND_ABDUCTION_OF_OTHERS",
    "IPC_KIDNAPPING_ABDUCTION",
    "IPC_KIDNAPPING_FOR_RANSOM",

    "IPC_ARSON",

    "IPC_ATTEMPT_TO_MURDER",
]

TARGET_SOURCE_COLUMNS = [
    col for col in TARGET_SOURCE_COLUMNS
    if col in df.columns
]

print(
    "Target-source columns found:",
    len(TARGET_SOURCE_COLUMNS)
)

for col in TARGET_SOURCE_COLUMNS:
    print("-", col)

Target-source columns found: 28
- IPC_CULPABLE_HOMICIDE_NOT_AMOUNTING_TO_MURDER
- IPC_RIOTS
- IPC_RIOTS_COMMUNAL
- IPC_RIOTS_INDUSTRIAL
- IPC_RIOTS_POLITICAL
- IPC_RIOTS_CASTE_CONFLICT
- IPC_RIOTS_AGRARIAN
- IPC_RIOTS_STUDENTS
- IPC_RIOTS_SECTARIAN
- IPC_RAPE
- IPC_OTHER_RAPE
- IPC_CUSTODIAL_RAPE
- IPC_CUSTODIAL_GANG_RAPE
- IPC_RAPE_GANG_RAPE
- IPC_RAPE_OTHERS
- IPC_MURDER
- IPC_PREPARATION_AND_ASSEMBLY_FOR_DACOITY
- IPC_DOWRY_DEATHS
- IPC_ROBBERY
- IPC_DACOITY
- IPC_OTHER_DACOITY
- IPC_DACOITY_WITH_MURDER
- IPC_KIDNAPPING_AND_ABDUCTION_OF_WOMEN_AND_GIRLS
- IPC_KIDNAPPING_AND_ABDUCTION_OF_OTHERS
- IPC_KIDNAPPING_ABDUCTION
- IPC_KIDNAPPING_FOR_RANSOM
- IPC_ARSON
- IPC_ATTEMPT_TO_MURDER


In [5]:
EXCLUDED_COLUMNS = set(
    TARGET_SOURCE_COLUMNS
    + [
        TARGET,
        "VIOLENT_CRIME_BURDEN",
    ]
)

# Engineered versions of the same violent-crime groups
EXCLUDED_COLUMNS.update(
    [
        col
        for col in df.columns
        if col.startswith("GROUPED_")
        and any(
            key in col
            for key in [
                "CHNOT",
                "RIOTS",
                "RAPE",
                "MURDER",
                "DACOITY",
                "DOWRY",
                "ROBBERY",
                "KIDNAPPING",
                "ARSON",
                "ATTEMPT_TO_MURDER",
            ]
        )
    ]
)

print("Excluded columns:", len(EXCLUDED_COLUMNS))

Excluded columns: 30


In [6]:
FEATURE_COLUMNS = [
    col
    for col in df.columns
    if col not in EXCLUDED_COLUMNS
]

X = df[FEATURE_COLUMNS].copy()
y = df[TARGET].copy()

print("Predictor columns:", len(FEATURE_COLUMNS))
print("Target:", TARGET)
print("X shape:", X.shape)
print("y shape:", y.shape)

Predictor columns: 303
Target: TARGET_VIOLENT_LEVEL
X shape: (9631, 303)
y shape: (9631,)


In [7]:
missing_percentage = (
    X.isna()
    .mean() * 100
)

MAX_MISSING_PERCENTAGE = 80

usable_features = (
    missing_percentage[
        missing_percentage <= MAX_MISSING_PERCENTAGE
    ]
    .index
    .tolist()
)

X = X[usable_features].copy()

print(
    "Features before missingness filter:",
    len(FEATURE_COLUMNS)
)

print(
    "Features after missingness filter:",
    len(usable_features)
)

print(
    "Removed:",
    len(FEATURE_COLUMNS) - len(usable_features)
)

print(
    "Final X shape:",
    X.shape
)

Features before missingness filter: 303
Features after missingness filter: 62
Removed: 241
Final X shape: (9631, 62)


In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))

print("\nTraining distribution:")
print(
    y_train.value_counts()
    .sort_index()
)

print("\nTesting distribution:")
print(
    y_test.value_counts()
    .sort_index()
)

Training samples: 7704
Testing samples : 1927

Training distribution:
TARGET_VIOLENT_LEVEL
1    1541
2    1541
3    1541
4    1540
5    1541
Name: count, dtype: int64

Testing distribution:
TARGET_VIOLENT_LEVEL
1    386
2    385
3    385
4    386
5    385
Name: count, dtype: int64


In [9]:
numeric_features = (
    X_train
    .select_dtypes(
        include=np.number
    )
    .columns
    .tolist()
)

categorical_features = (
    X_train
    .select_dtypes(
        include=["object", "string", "category"]
    )
    .columns
    .tolist()
)

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))

Numeric features: 55
Categorical features: 2


In [10]:
numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "encoder",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ],
    remainder="drop"
)

print("Preprocessor created successfully.")

Preprocessor created successfully.


In [11]:
nb_model = Pipeline(
    steps=[
        (
            "preprocessing",
            preprocessor
        ),
        (
            "classifier",
            GaussianNB()
        )
    ]
)

nb_model.fit(
    X_train,
    y_train
)

nb_pred = nb_model.predict(
    X_test
)

nb_prob = nb_model.predict_proba(
    X_test
)

print("Naive Bayes training completed.")

Naive Bayes training completed.


In [12]:
random_tree_model = Pipeline(
    steps=[
        (
            "preprocessing",
            preprocessor
        ),
        (
            "classifier",
            ExtraTreeClassifier(
                random_state=42,
                max_features="sqrt"
            )
        )
    ]
)

random_tree_model.fit(
    X_train,
    y_train
)

random_tree_pred = (
    random_tree_model.predict(
        X_test
    )
)

random_tree_prob = (
    random_tree_model.predict_proba(
        X_test
    )
)

print("Random Tree training completed.")

Random Tree training completed.


In [13]:
multiclass_model = Pipeline(
    steps=[
        (
            "preprocessing",
            preprocessor
        ),
        (
            "classifier",
            OneVsRestClassifier(
                GaussianNB()
            )
        )
    ]
)

multiclass_model.fit(
    X_train,
    y_train
)

multiclass_pred = (
    multiclass_model.predict(
        X_test
    )
)

multiclass_prob = (
    multiclass_model.predict_proba(
        X_test
    )
)

print("Multi-class classifier training completed.")

Multi-class classifier training completed.


In [15]:
def evaluate_model(
    name,
    y_true,
    y_pred,
    y_prob
):
    result = {
        "Model": name,

        "Accuracy": accuracy_score(
            y_true,
            y_pred
        ),

        "Precision": precision_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        ),

        "Recall": recall_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        ),

        "F1": f1_score(
            y_true,
            y_pred,
            average="weighted",
            zero_division=0
        ),

        "ROC_AUC": np.nan,

        "PR_AUC": np.nan
    }

    # ---------------------------------------------------------------
    # Check probability output
    # ---------------------------------------------------------------

    if y_prob is not None:

        y_prob = np.asarray(y_prob)

        has_nan = np.isnan(y_prob).any()
        has_inf = np.isinf(y_prob).any()

        if not has_nan and not has_inf:

            try:
                result["ROC_AUC"] = roc_auc_score(
                    y_true,
                    y_prob,
                    multi_class="ovr",
                    average="weighted"
                )

                result["PR_AUC"] = (
                    average_precision_score(
                        y_true,
                        y_prob,
                        average="weighted"
                    )
                )

            except ValueError as e:

                print(
                    f"{name}: ROC-AUC/PR-AUC could not be calculated."
                )

                print(
                    "Reason:",
                    e
                )

        else:

            print(
                f"{name}: probability output contains NaN/Inf."
            )

            print(
                "ROC-AUC and PR-AUC set to NaN."
            )

    return result


# =====================================================================
# EVALUATE ALL MODELS
# =====================================================================

results = []

results.append(
    evaluate_model(
        "Naive Bayes",
        y_test,
        nb_pred,
        nb_prob
    )
)

results.append(
    evaluate_model(
        "Random Tree",
        y_test,
        random_tree_pred,
        random_tree_prob
    )
)

results.append(
    evaluate_model(
        "Multi-class Classifier",
        y_test,
        multiclass_pred,
        multiclass_prob
    )
)

results_df = pd.DataFrame(
    results
)

print("\nFINAL MODEL RESULTS")
print("=" * 70)

display(
    results_df.round(4)
)

Multi-class Classifier: probability output contains NaN/Inf.
ROC-AUC and PR-AUC set to NaN.

FINAL MODEL RESULTS


,Model,Accuracy,Precision,Recall,F1,ROC_AUC,PR_AUC
0,Naive Bayes,0.6248,0.6310,0.6248,0.5962,0.8740,0.5809
1,Random Tree,0.6165,0.6118,0.6165,0.6137,0.7603,0.4742
2,Multi-class Classifier,0.4966,0.5351,0.4966,0.4797,NaN,NaN


In [16]:
results_path = (
    TABLES_DIR
    / "basepaper_model_results.csv"
)

results_df.to_csv(
    results_path,
    index=False
)

print(
    "Results saved to:",
    results_path
)

Results saved to: d:\Major_Project\Crime_Analysis\outputs\tables\basepaper_model_results.csv
